# NumPy로 GPT-2 구현: 실험과 그림 재현

블로그 글 [NumPy로 GPT-2 구현하며 LLM 동작 원리 이해하기](https://grace0409.github.io/posts/numpy-gpt2-from-scratch/)의 실험을 순서대로 실행하고, 글에 쓴 그림 6개를 다시 그립니다.

- 위에서부터 차례로 실행하면 됩니다. CPU 런타임으로 충분하고, 6번의 학습 셀은 GPU가 있으면 더 빠릅니다.
- 오래 걸리는 셀에는 예상 시간을 적었습니다.

In [ ]:
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/Grace0409/grace0409.github.io.git blog
%cd blog
!git sparse-checkout set gpt2-numpy
%cd gpt2-numpy

In [ ]:
!pip -q install tiktoken safetensors regex transformers
!apt-get -qq install -y fonts-nanum

## 0. 데이터 내려받기

Hugging Face의 GPT-2 가중치(약 548MB)와 Tiny Shakespeare를 내려받고, 가중치는 공개된 해시와 대조합니다.

In [ ]:
!python experiments/download_data.py

## 1. 토크나이저 (2단계)

한국어와 영어의 토큰 수를 비교하고, 직접 만든 BPE를 tiktoken과 대조합니다. BPE 학습이 있어 몇 분 걸립니다.

In [ ]:
!python experiments/e1_tokenizer.py

## 2. Hugging Face와 logits 비교 (5~6단계)

float32와 float64로 최대 오차를 잽니다.

In [ ]:
!python experiments/e2_verify_hf.py

## 3. 임베딩과 어텐션 (3~4단계)

In [ ]:
!python experiments/e3_attention.py

## 4. 샘플링 (7단계)

greedy, temperature(0.1, 1.0, 2.0), top-k, top-p를 프롬프트마다 64개씩 생성해 비교합니다. 가장 오래 걸리는 셀입니다.

In [ ]:
!python experiments/e4_sampling.py

## 5. 부품 제거: 학습된 GPT-2 (8단계)

√d_k, causal mask, 잔차 연결, 위치 임베딩을 하나씩 끄고 손실을 잽니다. √d_k를 난수로 따로 재는 실험도 들어 있습니다.

In [ ]:
!python experiments/e5_ablation.py

## 6. 부품 제거: 처음부터 학습 (8단계)

전체 실험은 학습 36회이고 CPU 12코어로 16시간 가까이 걸렸습니다. 그래서 저장된 결과(`results/e6_runs`)를 읽어 요약하고, 학습 코드가 도는지는 4가지 변형을 300스텝만 돌려 확인합니다.

In [ ]:
!python experiments/e6_train_ablation.py --summary-only

In [ ]:
!python experiments/e6_train_ablation.py --quick

## 7. 손실과 파인튜닝 (9단계)

사전학습, SFT, RLHF(PPO), DPO 손실을 계산하고 같은 가중치에서 30스텝씩 파인튜닝합니다.

In [ ]:
!python experiments/e7_losses.py

## 8. 그림 그리기

저장된 결과로 글에 쓴 그림 6개를 그립니다.

In [ ]:
!python experiments/make_figures.py

In [ ]:
from IPython.display import Image, display

for name in ["token_counts", "attention_map", "temperature", "train_ablation", "depth", "fine_tuning"]:
    display(Image(f"figures/{name}.png", width=720))

## 9. 그림 내려받기

In [ ]:
!zip -qr figures.zip figures
from google.colab import files

files.download("figures.zip")